In [8]:
# Load libraries 
import pandas as pd
import numpy as np 
from math import sqrt 
import sys 
print(sys.executable)
import statsmodels.api as sm
from scipy.stats import norm

/opt/anaconda3/envs/btc_tf/bin/python


In [9]:
# Load all predictions CV for all the models here 
LSTM_PRED_PATH = "../forecast evaluations/lstm_outputs/lstm_rolling_oos_predictions.csv"
HAR_PRED_PATH  = "../forecast evaluations/har_outputs/har_family_rolling_oos_predictions.csv"
LASSO_PRED_PATH = "../forecast evaluations/lasso_outputs/lasso_rolling_oos_predictions.csv"
RIDGE_PRED_PATH = "../forecast evaluations/ridge_outputs/ridge_rolling_oos_predictions.csv"
SVR_PRED_PATH = "../forecast evaluations/svr_outputs/svr_rolling_oos_predictions.csv"
XGB_PRED_PATH = "../forecast evaluations/xgb_outputs/xgb_rolling_oos_predictions.csv"
RF_PRED_PATH = "../forecast evaluations/rf_outputs/rf_rolling_oos_predictions.csv"

 
BENCHMARK = "HAR-RV"
HORIZONS = [1, 3, 5, 7]

In [10]:
# Load paths
def load_lstm_preds(path):
    df = pd.read_csv(path)
    # expected: date, h, y_true, y_pred (may also include error)
    df = df.copy()
    df["model"] = "LSTM"
    df["date"] = pd.to_datetime(df["date"],dayfirst=True, errors = "raise")
    df["h"] = df["h"].astype(int)
    df = df.rename(columns={"y_true": "actual", "y_pred": "predicted"})
    keep = ["date", "h", "model", "actual", "predicted"]
    return df[keep]


def load_preds(path):
    df = pd.read_csv(path)
    # expected: date, model, h, y_true, y_pred
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"],dayfirst=True, errors = "raise")
    df["h"] = df["h"].astype(int)
    df = df.rename(columns={"y_true": "actual", "y_pred": "predicted"})
    keep = ["date", "h", "model", "actual", "predicted"]
    return df[keep]

In [11]:
def dm_test_hac(loss_model, loss_bench, hac_lag):
    d = np.asarray(loss_model - loss_bench, dtype=float)
    d = d[~np.isnan(d)]
    T = len(d)

    if T < 20:
        return {"n": T, "dm_stat": np.nan, "p_value": np.nan, "mean_d": np.nan}

    X = np.ones((T, 1))
    res = sm.OLS(d, X).fit(cov_type="HAC", cov_kwds={"maxlags": int(hac_lag)})
    dm_stat = float(res.tvalues[0])
    p_value = round(float(2.0 * (1.0 - norm.cdf(abs(dm_stat)))),6)  # two-sided
    mean_d = float(np.mean(d))

    return {"n": T, "dm_stat": dm_stat, "p_value": p_value, "mean_d": mean_d}


In [12]:
def dm_table_vs_benchmark(preds_long, benchmark=BENCHMARK, horizons=HORIZONS):
    if benchmark not in preds_long["model"].unique():
        raise ValueError(f"Benchmark '{benchmark}' not found in preds_long['model'].")

    out_rows = []
    models = sorted([m for m in preds_long["model"].unique() if m != benchmark])

    for h in horizons:
        bench = preds_long[(preds_long["model"] == benchmark) & (preds_long["h"] == h)][
            ["date", "actual", "predicted"]
        ].rename(columns={"predicted": "pred_b", "actual": "actual_b"})

        if bench.empty:
            continue

        for m in models:
            other = preds_long[(preds_long["model"] == m) & (preds_long["h"] == h)][
                ["date", "actual", "predicted"]
            ].rename(columns={"predicted": "pred_m"})

            merged = bench.merge(other, on=["date"], how="inner")

            # squared error loss (RMSE objective)
            actual = merged["actual_b"]
            loss_m = (actual - merged["pred_m"]) ** 2
            loss_b = (actual - merged["pred_b"]) ** 2

            hac_lag = max(h - 1, 0)

            res = dm_test_hac(loss_m, loss_b, hac_lag=hac_lag)

            # Interpretation:
            # mean_d = mean(loss_m - loss_b)
            # if mean_d < 0 => model m has smaller loss => favours m
            favours = m if (res["mean_d"] < 0) else benchmark

            out_rows.append({
                "h": h,
                "model": m,
                "benchmark": benchmark,
                "hac_lag": hac_lag,
                "dm_stat": res["dm_stat"],
                "p_value": res["p_value"],
                'sig_5pct': 'Yes' if res["p_value"] < 0.05 else 'No',
                "favours": favours
            })

    return pd.DataFrame(out_rows).sort_values(["h", "model"]).reset_index(drop=True)


In [13]:
lstm_df = load_lstm_preds(LSTM_PRED_PATH)
har_df = load_preds(HAR_PRED_PATH)
svr_df = load_preds(SVR_PRED_PATH)
xgb_df = load_preds(XGB_PRED_PATH)
lasso_df = load_preds(LASSO_PRED_PATH)
ridge_df = load_preds(RIDGE_PRED_PATH)
rf_df = load_preds(RF_PRED_PATH)

# print(lstm_df.head(3))
# print(har_df.head(3))
# print(svr_df.head(3))
# print(xgb_df.head(3))
# print(lasso_df.head(3))
# print(ridge_df.head(3))
# print(rf_df.head(3))

/var/folders/56/jfcfpz9j7jddpghsmvgv_mfr0000gn/T/ipykernel_9554/1920358959.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["date"],dayfirst=True, errors = "raise")
/var/folders/56/jfcfpz9j7jddpghsmvgv_mfr0000gn/T/ipykernel_9554/1920358959.py:18: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["date"] = pd.to_datetime(df["date"],dayfirst=True, errors = "raise")
/var/folders/56/jfcfpz9j7jddpghsmvgv_mfr0000gn/T/ipykernel_9554/1920358959.py:18: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["date"] = pd.to_datetime(df["date"],dayfirst=True, errors = "raise")
/var/folders/56/jfcfpz9j7jddpghsmvgv_mfr0000gn/T/ipyker

In [14]:
preds = pd.concat([lstm_df, har_df, svr_df, xgb_df,lasso_df,ridge_df, rf_df], ignore_index=True)

# Basic sanity checks
preds = preds.dropna(subset=["date", "h", "model", "actual", "predicted"])
preds = preds.sort_values(["h", "model", "date"]).reset_index(drop=True)

dm_results = dm_table_vs_benchmark(preds, benchmark=BENCHMARK, horizons=HORIZONS)
print("\nDiebold-Mariano test vs benchmark-", BENCHMARK)
print(dm_results)


Diebold-Mariano test vs benchmark- HAR-RV
    h         model benchmark  hac_lag   dm_stat   p_value sig_5pct  \
0   1      HAR-RV-J    HAR-RV        0 -3.111166  0.001864      Yes   
1   1    HAR-RV-J-H    HAR-RV        0 -4.162568  0.000031      Yes   
2   1         LASSO    HAR-RV        0 -6.368450  0.000000      Yes   
3   1          LSTM    HAR-RV        0 -0.280776  0.778882       No   
4   1         RIDGE    HAR-RV        0 -4.874310  0.000001      Yes   
5   1  RandomForest    HAR-RV        0 -7.468027  0.000000      Yes   
6   1           SVR    HAR-RV        0 -5.041380  0.000000      Yes   
7   1       XGBoost    HAR-RV        0 -7.695873  0.000000      Yes   
8   3      HAR-RV-J    HAR-RV        2 -1.289544  0.197209       No   
9   3    HAR-RV-J-H    HAR-RV        2 -0.767728  0.442649       No   
10  3         LASSO    HAR-RV        2  0.511173  0.609230       No   
11  3          LSTM    HAR-RV        2  4.625982  0.000004      Yes   
12  3         RIDGE    HAR-RV     